INSTALL

In [2]:
%pip install pandas numpy torch torchvision Pillow tqdm scikit-learn xgboost

  Using cached scikit_learn-1.8.0-cp311-cp311-win_amd64.whl.metadata (11 kB)
  Using cached xgboost-3.2.0-py3-none-win_amd64.whl.metadata (2.1 kB)
  Using cached scipy-1.17.1-cp311-cp311-win_amd64.whl.metadata (60 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached scikit_learn-1.8.0-cp311-cp311-win_amd64.whl (8.1 MB)
Using cached xgboost-3.2.0-py3-none-win_amd64.whl (101.7 MB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached scipy-1.17.1-cp311-cp311-win_amd64.whl (36.6 MB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: C:\Users\TheRe\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


IMPORTS

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, TensorDataset
from PIL import Image
import os as os
import tqdm as tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from xgboost import XGBClassifier

PREPROCESSING

In [ ]:
t1_train_dir = r'Data\task1_data\images\train'
t1_test_dir = r'Data\task1_data\images\test'

if not os.path.exists(t1_train_dir):
    raise FileNotFoundError(f"Training directory not found: {t1_train_dir}")

if not os.path.exists(t1_test_dir):
    raise FileNotFoundError(f"Test directory not found: {t1_test_dir}")


SAVE PREPROCESSED IMAGES

In [ ]:
# ResNet requires 224x224 inputs.
# Mean and std values are from ImageNet (the dataset ResNet was trained on).
# Normalising to the same distribution ResNet expects ensures meaningful feature extraction.
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

def image_to_resnet(image_dir, output_path, has_labels=True):
    tensors = []
    labels = []
    paths = []

    if has_labels:
        # subfolder names are the class labels
        class_names = sorted(os.listdir(image_dir))
        class_to_idx = {cls: idx for idx, cls in enumerate(class_names)}

        # count total images for progress display
        total = sum(
            len(os.listdir(os.path.join(image_dir, c)))
            for c in class_names
            if os.path.isdir(os.path.join(image_dir, c))
        )

        c = 0
        for class_name in class_names:
            class_dir = os.path.join(image_dir, class_name)
            if not os.path.isdir(class_dir):
                continue
            for fname in os.listdir(class_dir):
                if not fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                    continue
                img = Image.open(os.path.join(class_dir, fname)).convert("RGB")
                tensors.append(transform(img))
                labels.append(class_to_idx[class_name])
                paths.append(os.path.join(class_dir, fname))
                c += 1
                print(f'{c}/{total}', end='\r')

        torch.save({
            'tensors': torch.stack(tensors),
            'labels': torch.tensor(labels),
            'paths': paths,
            'class_to_idx': class_to_idx
        }, output_path)

    else:
        # test directory is flat = no subfolders, no labels
        fnames = [f for f in os.listdir(image_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        total = len(fnames)
        for c, fname in enumerate(fnames, 1):
            img_path = os.path.join(image_dir, fname)
            img = Image.open(img_path).convert("RGB")
            tensors.append(transform(img))
            paths.append(img_path)
            print(f'{c}/{total}', end='\r')

        torch.save({
            'tensors': torch.stack(tensors),
            'paths': paths
        }, output_path)

    print(f"\nSaved {len(tensors)} images to {output_path}")

image_to_resnet(t1_train_dir, r'Data\task1_data\t1_train_resnet.pt', has_labels=True)
image_to_resnet(t1_test_dir,  r'Data\task1_data\t1_test_resnet.pt',  has_labels=False)


3750/3750
Saved 3750 images to Data\task1_data\t1_train_resnet.pt
1250/1250
Saved 1250 images to Data\task1_data\t1_test_resnet.pt


RESNET FEATURE EXTRACTION

In [ ]:
# load ResNet50, strip the final classification layer to get a 2048-d feature extractor
resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
resnet = nn.Sequential(*list(resnet.children())[:-1])
resnet.eval()

def extract_resnet_features(pt_path, output_path):
    data = torch.load(pt_path)
    tensors = data['tensors']   # shape: (N, 3, 224, 224)

    features = []
    total = len(tensors)

    with torch.no_grad():
        for i, img_tensor in enumerate(tensors, 1):
            feat = resnet(img_tensor.unsqueeze(0))  # (1, 2048, 1, 1)
            feat = feat.squeeze().numpy()           # (2048,)
            features.append(feat)
            print(f'{i}/{total}', end='\r')

    features = np.array(features)  # (N, 2048)

    save_data = {**data, 'features': features}
    torch.save(save_data, output_path)
    print(f"\nExtracted features shape: {features.shape} -> saved to {output_path}")

extract_resnet_features(r'Data\task1_data\t1_train.pt', r'Data\task1_data\t1_train_resnet.pt')
extract_resnet_features(r'Data\task1_data\t1_test.pt',  r'Data\task1_data\t1_test_resnet.pt')

CLASSIFICATION MODELS

In [ ]:
"""
So, we're going to use some basic models:
1. Linear Classifier
2. SVM
3. XGBoost
4. MLP networks --> we could try different activations and different layers
"""

def _compute_metrics(y_true, y_pred):
    classes, counts = np.unique(y_true, return_counts=True)
    balance = {int(c): round(100 * n / len(y_true), 1) for c, n in zip(classes, counts)}
    return {
        'accuracy':      accuracy_score(y_true, y_pred),
        'f1':            f1_score(y_true, y_pred, average='weighted'),
        'precision':     precision_score(y_true, y_pred, average='weighted'),
        'recall':        recall_score(y_true, y_pred, average='weighted'),
        'class_balance': balance,
    }

class MLP(nn.Module):
    def __init__(self, input_dim: int, activation=nn.GELU, hidden=(64, 32), dropout=0.0):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden:
            layers.append(nn.Linear(prev, h))
            try:
                layers.append(nn.GELU(approximate="tanh") if activation == nn.GELU else activation(h))
            except TypeError:
                layers.append(activation())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

def train_linear_classifier(X_train, y_train, X_test, y_test):
    clf = LogisticRegression()
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    return clf, _compute_metrics(y_test, y_pred)

def train_svm(X_train, y_train, X_test, y_test, kernel='linear'):
    clf = SVC(kernel=kernel)
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    return clf, _compute_metrics(y_test, y_pred)

def train_xgboost(X_train, y_train, X_test, y_test):
    clf = XGBClassifier(use_label_encoder=False, eval_metric='mlogloss')
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    return clf, _compute_metrics(y_test, y_pred)

def train_mlp(X_train, y_train, X_test, y_test,
              hidden=(64, 32), activation=nn.GELU, dropout=0.0,
              epochs=512, patience=100, lr=1e-3, device=None):

    # i have a GPU, but you may not. so the net will run on GPU if available, otherwise CPU. it will be slower on CPU, but should still work.
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")

    # convert numpy arrays to PyTorch tensors and move to device
    X_tr = torch.tensor(np.asarray(X_train, dtype=np.float32), device=device)
    y_tr = torch.tensor(np.asarray(y_train, dtype=np.float32), device=device).unsqueeze(1)
    X_te = torch.tensor(np.asarray(X_test,  dtype=np.float32), device=device)

    # loader is in charge of batching and shuffling the training data.
    loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=256, shuffle=True)
    
    # create the MLP model and move it to the device
    model = MLP(input_dim=X_tr.shape[1], activation=activation, hidden=hidden, dropout=dropout).to(device)
    
    # we use Adam optimizer and binary cross-entropy loss with logits (since the model outputs raw scores and we have a binary classification task)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.BCEWithLogitsLoss()

    # we save the best model state if we didn't end up improving at the end of epochs
    best_loss, best_state, wait = float("inf"), None, 0

    # train loop
    for _ in range(epochs):
        model.train() # set model to training mode (enables dropout if used) <- btw drop out means randomly ignoring some neurons during training to prevent overfitting
        for X_batch, y_batch in loader:
            optimizer.zero_grad() # clears old grads from the last step
            criterion(model(X_batch), y_batch).backward() # pass batch through, computes loss, and calculates grads
            optimizer.step() # update weights

        model.eval() # switch to eval mode for validation (disables dropout)
        with torch.no_grad():
            train_loss = criterion(model(X_tr), y_tr).item() # gets the loss

        # ensures we get the best model state 
        if train_loss < best_loss:
            best_loss, best_state, wait = train_loss, model.state_dict(), 0
        else:
            # stop training if we dont improve for a while (patience)
            wait += 1
            if wait >= patience:
                break

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad(): # run the predictions on the test set
        y_pred = (torch.sigmoid(model(X_te).squeeze(1)) > 0.5).long().cpu().numpy()

    return model, _compute_metrics(np.asarray(y_test), y_pred)